## AB Testing Models - Evaluating Agent Performance

Learn how to systematically evaluate multiple agent variants using different language models. This tutorial shows you how to run parallel evaluations, compare performance metrics, analyze cost-performance trade-offs, and generate data-driven model selection recommendations.

### What You'll Learn
- Load agent configurations from Part 1 and recreate agents
- Create diverse evaluation datasets from TauBench scenarios
- Run systematic evaluations on multiple agent variants
- Use five built-in evaluators for comprehensive assessment
- Compare evaluation scores side-by-side across models
- Calculate statistical measures (mean, standard deviation)
- Analyze cost-performance trade-offs
- Generate actionable model selection recommendations

### Tutorial Details

| Information         | Details                                                                       |
|:--------------------|:------------------------------------------------------------------------------|
| Tutorial type       | Advanced - Systematic model evaluation and comparison                         |
| Tutorial components | Multi-model evaluation, statistical analysis, cost-performance optimization   |
| Tutorial vertical   | Agent Evaluation                                                              |
| Example complexity  | Advanced                                                                      |
| SDK used            | Strands Agents, Strands Evals                                                 |

### Understanding A/B Testing Evaluation

Systematic evaluation answers: Which model produces highest quality responses? Best completes goals? Selects tools most accurately? Provides best cost-performance ratio?

#### Five Dimensions of Evaluation

| Evaluator | Purpose |
|:----------|:--------|
| OutputEvaluator | Measures response quality, correctness, completeness |
| HelpfulnessEvaluator | Assesses practical value, clarity, actionability |
| GoalSuccessRateEvaluator | Tracks binary success/failure for task completion |
| ToolSelectionAccuracyEvaluator | Validates appropriate tool choices |
| ToolParameterAccuracyEvaluator | Checks parameter correctness and formatting |

#### Evaluation Methodology

| Requirement | Description |
|:------------|:------------|
| Same test dataset | All models tested on identical queries |
| Same evaluation criteria | Consistent scoring across variants |
| Multiple test cases | Statistical validity through volume |

**Analysis Flow**: Run agents → Collect scores → Calculate statistics → Compare side-by-side → Factor cost → Generate recommendations

### Environment Setup

Set `OPENAI_API_KEY` in your environment (optionally `OPENAI_MODEL_ID` for the judge, default `gpt-5-mini`) and prepare for multi-model evaluation.

In [ ]:
import os
from strands.models.openai import OpenAIModel

# OpenAI judge model for all evaluators (reads OPENAI_API_KEY from the environment)
JUDGE_MODEL = OpenAIModel(
    client_args={"api_key": os.environ["OPENAI_API_KEY"]},
    model_id=os.getenv("OPENAI_MODEL_ID", "gpt-5-mini"),
)

### Setup and Imports

Import all necessary libraries for evaluation and statistical analysis.

In [ ]:
import os
import sys
import json
import logging
from typing import Dict, List, Any
import statistics
from collections import defaultdict

# Add paths for airline tools (local data directory)
sys.path.append('./data/ma-bench/')
sys.path.append('./data/tau-bench/')

# Strands imports
from strands import Agent

# Strands Evals imports
from strands_evals import Experiment, Case
from strands_evals.evaluators import (
    OutputEvaluator,
    HelpfulnessEvaluator,
    GoalSuccessRateEvaluator,
    ToolSelectionAccuracyEvaluator,
    ToolParameterAccuracyEvaluator
)

# Display utilities
from IPython.display import Markdown, display

# Disable verbose logging
logging.basicConfig(level=logging.CRITICAL)
for logger_name in ["strands", "graph", "event_loop", "registry", "sliding_window_conversation_manager", "streaming"]:
    logging.getLogger(logger_name).setLevel(logging.CRITICAL)

# Bypass tool consent
os.environ["BYPASS_TOOL_CONSENT"] = "true"

### Load Agent Configurations

Load the agent configurations saved in Part 1 (Tutorial 07a).

In [ ]:
# Load configuration from Part 1
config_path = "./agent_configs.json"

with open(config_path, "r") as f:
    config = json.load(f)

print("Loaded agent configurations:")
print(f"  Models: {len(config['models'])}")
print(f"  Dataset path: {config['dataset_path']}")
print(f"  Tools: {config['num_tools']}")
print("\nModel variants:")
for key, model_info in config['models'].items():
    print(f"  {model_info['name']}: {model_info['model_id']}")

### Import Airline Domain Tools

Recreate the airline tool environment for all three agent variants.

In [ ]:
# Import all 14 airline tools
from mabench.environments.airline.tools.book_reservation import book_reservation
from mabench.environments.airline.tools.calculate import calculate
from mabench.environments.airline.tools.cancel_reservation import cancel_reservation
from mabench.environments.airline.tools.get_reservation_details import get_reservation_details
from mabench.environments.airline.tools.get_user_details import get_user_details
from mabench.environments.airline.tools.list_all_airports import list_all_airports
from mabench.environments.airline.tools.search_direct_flight import search_direct_flight
from mabench.environments.airline.tools.search_onestop_flight import search_onestop_flight
from mabench.environments.airline.tools.send_certificate import send_certificate
from mabench.environments.airline.tools.think import think
from mabench.environments.airline.tools.transfer_to_human_agents import transfer_to_human_agents
from mabench.environments.airline.tools.update_reservation_baggages import update_reservation_baggages
from mabench.environments.airline.tools.update_reservation_flights import update_reservation_flights
from mabench.environments.airline.tools.update_reservation_passengers import update_reservation_passengers

# Import airline policy
from tau_bench.envs.airline.wiki import WIKI

# Define tools list
AIRLINE_TOOLS = [
    book_reservation,
    calculate,
    cancel_reservation,
    get_reservation_details,
    get_user_details,
    list_all_airports,
    search_direct_flight,
    search_onestop_flight,
    send_certificate,
    think,
    transfer_to_human_agents,
    update_reservation_baggages,
    update_reservation_flights,
    update_reservation_passengers,
]

# Create system prompt
SYSTEM_PROMPT_TEMPLATE = """
You are a helpful assistant for a travel website. Help the user answer any questions.

<instructions>
- Remember to check if the airport city is in the state mentioned by the user. For example, Houston is in Texas.
- Infer about the U.S. state in which the airport city resides. For example, Houston is in Texas.
- You should not use made-up or placeholder arguments.
</instructions>

<policy>
{policy}
</policy>
"""

AIRLINE_SYSTEM_PROMPT = SYSTEM_PROMPT_TEMPLATE.replace("{policy}", WIKI)

print(f"Imported {len(AIRLINE_TOOLS)} airline tools")
print(f"System prompt created ({len(AIRLINE_SYSTEM_PROMPT)} characters)")

### Recreate All Three Agent Variants

Build the three agent variants (GPT-5 nano, GPT-5, GPT-5 mini) for evaluation.

In [ ]:
# Create OpenAIModels for each variant
model_nano = OpenAIModel(
    client_args={"api_key": os.environ["OPENAI_API_KEY"]},
    model_id=config['models']['nano']['model_id']
)

model_gpt5 = OpenAIModel(
    client_args={"api_key": os.environ["OPENAI_API_KEY"]},
    model_id=config['models']['gpt5']['model_id']
)

model_mini = OpenAIModel(
    client_args={"api_key": os.environ["OPENAI_API_KEY"]},
    model_id=config['models']['mini']['model_id']
)

# Create agents
agent_nano = Agent(
    name="airline_assistant_nano",
    model=model_nano,
    tools=AIRLINE_TOOLS,
    system_prompt=AIRLINE_SYSTEM_PROMPT
)

agent_gpt5 = Agent(
    name="airline_assistant_gpt5",
    model=model_gpt5,
    tools=AIRLINE_TOOLS,
    system_prompt=AIRLINE_SYSTEM_PROMPT
)

agent_mini = Agent(
    name="airline_assistant_mini",
    model=model_mini,
    tools=AIRLINE_TOOLS,
    system_prompt=AIRLINE_SYSTEM_PROMPT
)

# Store in dictionary for easy iteration
agents = {
    "nano": {"agent": agent_nano, "name": "GPT-5 nano"},
    "gpt5": {"agent": agent_gpt5, "name": "GPT-5"},
    "mini": {"agent": agent_mini, "name": "GPT-5 mini"}
}

print("All three agent variants created:")
for key, info in agents.items():
    print(f"  {info['name']}: Ready")

### Load TauBench Dataset

Load the full TauBench airline dataset and select diverse test cases for evaluation.

In [ ]:
# Load TauBench dataset
dataset_path = config['dataset_path']

with open(dataset_path, "r") as file:
    all_tasks = json.load(file)

print(f"Loaded {len(all_tasks)} total scenarios from TauBench")
print("\nDataset includes:")
print("  - Flight searches and bookings")
print("  - Reservation modifications")
print("  - Cancellations and refunds")
print("  - Customer service inquiries")
print("  - Policy-based decisions")

### Select Diverse Test Cases

Choose 12 diverse test cases that represent different complexity levels and scenario types.

In [ ]:
# Select diverse test cases (indices chosen for variety)
selected_indices = [0, 1] #, 2, 4, 10, 15, 20, 25, 30, 35, 40, 45]
selected_tasks = [all_tasks[i] for i in selected_indices]

print(f"Selected {len(selected_tasks)} diverse test cases:")
print("\nTest Case Overview:")
for i, task in enumerate(selected_tasks, 1):
    question_preview = task['question'].replace('\n', ' ') + "..."
    print(f"  {i}. {question_preview}")

### Create Evaluation Dataset

Build Strands Evals dataset with test cases and five evaluators.

In [ ]:
# Create Case objects for evaluation
test_cases = []
for i, task in enumerate(selected_tasks):
    # Use 'question' if available, otherwise fall back to 'instruction'
    user_query = task.get('question', task.get('instruction', ''))
    case = Case(
        name=f"Scenario_{i+1}_{task['user_id']}",
        input=user_query,
        expected_output="A helpful, accurate response that addresses the user's airline service request."
    )
    test_cases.append(case)

# Create separate experiments for each evaluator
evaluator_list = [
    ("OutputEvaluator", OutputEvaluator(rubric="The output should provide a helpful, accurate response to the airline service request.", model=JUDGE_MODEL)),
    ("HelpfulnessEvaluator", HelpfulnessEvaluator(model=JUDGE_MODEL)),
    ("GoalSuccessRateEvaluator", GoalSuccessRateEvaluator(model=JUDGE_MODEL)),
    ("ToolSelectionAccuracyEvaluator", ToolSelectionAccuracyEvaluator(model=JUDGE_MODEL)),
    ("ToolParameterAccuracyEvaluator", ToolParameterAccuracyEvaluator(model=JUDGE_MODEL))
]

dataset_output = Experiment(
    cases=test_cases,
    evaluators=[evaluator_list[0][1]]
)

dataset_helpfulness = Experiment(
    cases=test_cases,
    evaluators=[evaluator_list[1][1]]
)

dataset_goal = Experiment(
    cases=test_cases,
    evaluators=[evaluator_list[2][1]]
)

dataset_tool_selection = Experiment(
    cases=test_cases,
    evaluators=[evaluator_list[3][1]]
)

dataset_tool_parameter = Experiment(
    cases=test_cases,
    evaluators=[evaluator_list[4][1]]
)

print(f"Evaluation experiments created:")
print(f"  Test cases: {len(test_cases)}")
print(f"  Evaluators: {len(evaluator_list)} (one experiment per evaluator)")
print("\nEvaluator types:")
for evaluator_name, _ in evaluator_list:
    print(f"  - {evaluator_name}")

### Run Evaluation: GPT-5 nano

Evaluate the GPT-5 nano agent variant on all test cases.

In [ ]:
# Define agent task function for GPT-5 nano
def agent_task_nano(case: Case) -> str:
    """Execute GPT-5 nano agent with the given case input."""
    agent_nano.messages = []
    response = agent_nano(case.input)
    return str(response)

# Run evaluations for all 5 evaluators
print("[EVALUATING GPT-5 NANO]")
print("Running evaluation on GPT-5 nano agent...")

print("  [1/5] OutputEvaluator...")
report_nano_output = dataset_output.run_evaluations(agent_task_nano)[0]

print("  [2/5] HelpfulnessEvaluator...")
report_nano_helpfulness = dataset_helpfulness.run_evaluations(agent_task_nano)[0]

print("  [3/5] GoalSuccessRateEvaluator...")
report_nano_goal = dataset_goal.run_evaluations(agent_task_nano)[0]

print("  [4/5] ToolSelectionAccuracyEvaluator...")
report_nano_tool_selection = dataset_tool_selection.run_evaluations(agent_task_nano)[0]

print("  [5/5] ToolParameterAccuracyEvaluator...")
report_nano_tool_parameter = dataset_tool_parameter.run_evaluations(agent_task_nano)[0]

print("GPT-5 nano evaluation complete (5 evaluation rounds)")

#### GPT-5 nano Results

Display evaluation results for GPT-5 nano.

In [ ]:
print("\n" + "=" * 80)
print("GPT-5 NANO RESULTS")
print("=" * 80)

print("\n[1/5] OUTPUT QUALITY")
print("-" * 80)
report_nano_output.display()

print("\n[2/5] HELPFULNESS")
print("-" * 80)
report_nano_helpfulness.display()

print("\n[3/5] GOAL SUCCESS RATE")
print("-" * 80)
report_nano_goal.display()

print("\n[4/5] TOOL SELECTION ACCURACY")
print("-" * 80)
report_nano_tool_selection.display()

print("\n[5/5] TOOL PARAMETER ACCURACY")
print("-" * 80)
report_nano_tool_parameter.display()

### Run Evaluation: GPT-5

Evaluate the GPT-5 agent variant on all test cases.

In [ ]:
# Define agent task function for GPT-5
def agent_task_gpt5(case: Case) -> str:
    """Execute GPT-5 agent with the given case input."""
    agent_gpt5.messages = []
    response = agent_gpt5(case.input)
    return str(response)

# Run evaluations for all 5 evaluators
print("[EVALUATING GPT-5]")
print("Running evaluation on GPT-5 agent...")

print("  [1/5] OutputEvaluator...")
report_gpt5_output = dataset_output.run_evaluations(agent_task_gpt5)[0]

print("  [2/5] HelpfulnessEvaluator...")
report_gpt5_helpfulness = dataset_helpfulness.run_evaluations(agent_task_gpt5)[0]

print("  [3/5] GoalSuccessRateEvaluator...")
report_gpt5_goal = dataset_goal.run_evaluations(agent_task_gpt5)[0]

print("  [4/5] ToolSelectionAccuracyEvaluator...")
report_gpt5_tool_selection = dataset_tool_selection.run_evaluations(agent_task_gpt5)[0]

print("  [5/5] ToolParameterAccuracyEvaluator...")
report_gpt5_tool_parameter = dataset_tool_parameter.run_evaluations(agent_task_gpt5)[0]

print("GPT-5 evaluation complete (5 evaluation rounds)")

#### GPT-5 Results

Display evaluation results for GPT-5.

In [ ]:
print("\n" + "=" * 80)
print("GPT-5 RESULTS")
print("=" * 80)

print("\n[1/5] OUTPUT QUALITY")
print("-" * 80)
report_gpt5_output.display()

print("\n[2/5] HELPFULNESS")
print("-" * 80)
report_gpt5_helpfulness.display()

print("\n[3/5] GOAL SUCCESS RATE")
print("-" * 80)
report_gpt5_goal.display()

print("\n[4/5] TOOL SELECTION ACCURACY")
print("-" * 80)
report_gpt5_tool_selection.display()

print("\n[5/5] TOOL PARAMETER ACCURACY")
print("-" * 80)
report_gpt5_tool_parameter.display()

### Run Evaluation: GPT-5 mini

Evaluate the GPT-5 mini agent variant on all test cases.

In [ ]:
# Define agent task function for GPT-5 mini
def agent_task_mini(case: Case) -> str:
    """Execute GPT-5 mini agent with the given case input."""
    agent_mini.messages = []
    response = agent_mini(case.input)
    return str(response)

# Run evaluations for all 5 evaluators
print("[EVALUATING GPT-5 MINI]")
print("Running evaluation on GPT-5 mini agent...")

print("  [1/5] OutputEvaluator...")
report_mini_output = dataset_output.run_evaluations(agent_task_mini)[0]

print("  [2/5] HelpfulnessEvaluator...")
report_mini_helpfulness = dataset_helpfulness.run_evaluations(agent_task_mini)[0]

print("  [3/5] GoalSuccessRateEvaluator...")
report_mini_goal = dataset_goal.run_evaluations(agent_task_mini)[0]

print("  [4/5] ToolSelectionAccuracyEvaluator...")
report_mini_tool_selection = dataset_tool_selection.run_evaluations(agent_task_mini)[0]

print("  [5/5] ToolParameterAccuracyEvaluator...")
report_mini_tool_parameter = dataset_tool_parameter.run_evaluations(agent_task_mini)[0]

print("GPT-5 mini evaluation complete (5 evaluation rounds)")

#### GPT-5 mini Results

Display evaluation results for GPT-5 mini.

In [ ]:
print("\n" + "=" * 80)
print("GPT-5 MINI RESULTS")
print("=" * 80)

print("\n[1/5] OUTPUT QUALITY")
print("-" * 80)
report_mini_output.display()

print("\n[2/5] HELPFULNESS")
print("-" * 80)
report_mini_helpfulness.display()

print("\n[3/5] GOAL SUCCESS RATE")
print("-" * 80)
report_mini_goal.display()

print("\n[4/5] TOOL SELECTION ACCURACY")
print("-" * 80)
report_mini_tool_selection.display()

print("\n[5/5] TOOL PARAMETER ACCURACY")
print("-" * 80)
report_mini_tool_parameter.display()

In [ ]:
# Helper function to extract scores from report
def extract_scores(report, evaluator_name: str) -> Dict[str, List[float]]:
    """Extract numeric scores organized by evaluator type.
    
    Args:
        report: EvaluationReport object containing detailed_results
        evaluator_name: Name of the evaluator (e.g., 'OutputEvaluator', 'HelpfulnessEvaluator')
    
    Returns:
        Dictionary mapping evaluator name to list of scores
    """
    scores = defaultdict(list)
    
    # detailed_results is a list of lists of EvaluationOutput objects
    # Each test case has its own list of EvaluationOutput objects
    for test_case_results in report.detailed_results:
        # test_case_results is a list of EvaluationOutput objects for one test case
        for eval_output in test_case_results:
            # eval_output is an EvaluationOutput object with fields: score, test_pass, reason, label
            if hasattr(eval_output, 'score'):
                scores[evaluator_name].append(float(eval_output.score))
    
    return dict(scores)

# Extract scores from all evaluation reports (3 models x evaluators)
scores_nano = {}
scores_nano.update(extract_scores(report_nano_output, 'OutputEvaluator'))
scores_nano.update(extract_scores(report_nano_helpfulness, 'HelpfulnessEvaluator'))
scores_nano.update(extract_scores(report_nano_goal, 'GoalSuccessRateEvaluator'))
scores_nano.update(extract_scores(report_nano_tool_selection, 'ToolSelectionAccuracyEvaluator'))
scores_nano.update(extract_scores(report_nano_tool_parameter, 'ToolParameterAccuracyEvaluator'))

scores_gpt5 = {}
scores_gpt5.update(extract_scores(report_gpt5_output, 'OutputEvaluator'))
scores_gpt5.update(extract_scores(report_gpt5_helpfulness, 'HelpfulnessEvaluator'))
scores_gpt5.update(extract_scores(report_gpt5_goal, 'GoalSuccessRateEvaluator'))
scores_gpt5.update(extract_scores(report_gpt5_tool_selection, 'ToolSelectionAccuracyEvaluator'))
scores_gpt5.update(extract_scores(report_gpt5_tool_parameter, 'ToolParameterAccuracyEvaluator'))

scores_mini = {}
scores_mini.update(extract_scores(report_mini_output, 'OutputEvaluator'))
scores_mini.update(extract_scores(report_mini_helpfulness, 'HelpfulnessEvaluator'))
scores_mini.update(extract_scores(report_mini_goal, 'GoalSuccessRateEvaluator'))
scores_mini.update(extract_scores(report_mini_tool_selection, 'ToolSelectionAccuracyEvaluator'))
scores_mini.update(extract_scores(report_mini_tool_parameter, 'ToolParameterAccuracyEvaluator'))

# Calculate totals dynamically
num_evaluators = len(scores_nano)
num_models = 3
total_reports = num_models * num_evaluators

print(f"Scores extracted from all {total_reports} evaluation reports")
print(f"  GPT-5 nano: {num_evaluators} evaluation rounds")
print(f"  GPT-5: {num_evaluators} evaluation rounds")
print(f"  GPT-5 mini: {num_evaluators} evaluation rounds")
print(f"\nEvaluator types found:")
for evaluator_name in scores_nano.keys():
    print(f"  - {evaluator_name}")

### Calculate Statistical Measures

Compute mean and standard deviation for each evaluator across all models.

In [ ]:
# Calculate statistics for each model
def calculate_statistics(scores: Dict[str, List[float]]) -> Dict[str, Dict[str, float]]:
    """Calculate mean and standard deviation for each evaluator."""
    stats = {}
    
    for evaluator_name, score_list in scores.items():
        if len(score_list) > 0:
            mean_score = statistics.mean(score_list)
            std_dev = statistics.stdev(score_list) if len(score_list) > 1 else 0.0
            stats[evaluator_name] = {
                "mean": mean_score,
                "std_dev": std_dev,
                "count": len(score_list)
            }
    
    return stats

# Calculate for all models
stats_nano = calculate_statistics(scores_nano)
stats_gpt5 = calculate_statistics(scores_gpt5)
stats_mini = calculate_statistics(scores_mini)

### Side-by-Side Comparison

Display evaluation scores side-by-side for direct comparison across models.

In [ ]:
# Create comparison table
comparison_md = "## Model Performance Comparison\n\n"
comparison_md += "### Mean Scores by Evaluator\n\n"
comparison_md += "| Evaluator | GPT-5 nano | GPT-5 | GPT-5 mini |\n"
comparison_md += "|:----------|-------------:|--------------:|----------:|\n"

# Get all evaluator names
all_evaluators = set(stats_nano.keys()) | set(stats_gpt5.keys()) | set(stats_mini.keys())

for evaluator_name in sorted(all_evaluators):
    nano_mean = stats_nano.get(evaluator_name, {}).get('mean', 0.0)
    gpt5_mean = stats_gpt5.get(evaluator_name, {}).get('mean', 0.0)
    mini_mean = stats_mini.get(evaluator_name, {}).get('mean', 0.0)
    
    # Highlight best score
    max_score = max(nano_mean, gpt5_mean, mini_mean)
    
    nano_str = f"**{nano_mean:.3f}**" if nano_mean == max_score else f"{nano_mean:.3f}"
    gpt5_str = f"**{gpt5_mean:.3f}**" if gpt5_mean == max_score else f"{gpt5_mean:.3f}"
    mini_str = f"**{mini_mean:.3f}**" if mini_mean == max_score else f"{mini_mean:.3f}"
    
    comparison_md += f"| {evaluator_name} | {nano_str} | {gpt5_str} | {mini_str} |\n"

comparison_md += "\n*Bold indicates best performance for that evaluator*\n"

display(Markdown(comparison_md))

### Statistical Variability Analysis

Examine the standard deviation to understand consistency across test cases.

In [ ]:
# Create variability table
variability_md = "## Performance Variability (Standard Deviation)\n\n"
variability_md += "| Evaluator | GPT-5 nano | GPT-5 | GPT-5 mini |\n"
variability_md += "|:----------|-------------:|--------------:|----------:|\n"

for evaluator_name in sorted(all_evaluators):
    nano_std = stats_nano.get(evaluator_name, {}).get('std_dev', 0.0)
    gpt5_std = stats_gpt5.get(evaluator_name, {}).get('std_dev', 0.0)
    mini_std = stats_mini.get(evaluator_name, {}).get('std_dev', 0.0)
    
    # Highlight lowest variability (most consistent)
    min_std = min(nano_std, gpt5_std, mini_std) if any([nano_std, gpt5_std, mini_std]) else 0
    
    nano_str = f"**{nano_std:.3f}**" if nano_std == min_std and min_std > 0 else f"{nano_std:.3f}"
    gpt5_str = f"**{gpt5_std:.3f}**" if gpt5_std == min_std and min_std > 0 else f"{gpt5_std:.3f}"
    mini_str = f"**{mini_std:.3f}**" if mini_std == min_std and min_std > 0 else f"{mini_std:.3f}"
    
    variability_md += f"| {evaluator_name} | {nano_str} | {gpt5_str} | {mini_str} |\n"

variability_md += "\n*Bold indicates lowest variability (most consistent performance)*\n"
variability_md += "\n**Lower standard deviation = More consistent performance across test cases**\n"

display(Markdown(variability_md))

### Cost-Performance Analysis

Analyze cost-performance trade-offs based on OpenAI API pricing.

In [ ]:
# Approximate OpenAI API pricing (per 1K tokens)
# Note: Actual pricing may change over time; check the OpenAI pricing page
pricing = {
    "nano": {"input": 0.00005, "output": 0.0004},   # GPT-5 nano pricing
    "gpt5": {"input": 0.00125, "output": 0.01},     # GPT-5 pricing
    "mini": {"input": 0.00025, "output": 0.002}     # GPT-5 mini pricing
}

# Create cost-performance summary
cost_md = "## Cost-Performance Trade-offs\n\n"
cost_md += "### Approximate Pricing (per 1K tokens)\n\n"
cost_md += "| Model | Input Cost | Output Cost | Relative Cost |\n"
cost_md += "|:------|:-----------|:------------|:--------------|\n"

# Calculate relative costs (normalized to GPT-5 nano)
nano_total = pricing["nano"]["input"] + pricing["nano"]["output"]
gpt5_total = pricing["gpt5"]["input"] + pricing["gpt5"]["output"]
mini_total = pricing["mini"]["input"] + pricing["mini"]["output"]

cost_md += f"| GPT-5 nano | ${pricing['nano']['input']:.5f} | ${pricing['nano']['output']:.5f} | 1.0x (baseline) |\n"
cost_md += f"| GPT-5 | ${pricing['gpt5']['input']:.5f} | ${pricing['gpt5']['output']:.5f} | {gpt5_total/nano_total:.1f}x |\n"
cost_md += f"| GPT-5 mini | ${pricing['mini']['input']:.5f} | ${pricing['mini']['output']:.5f} | {mini_total/nano_total:.1f}x |\n"

cost_md += "\n*Note: Pricing is approximate and may change over time*\n"

display(Markdown(cost_md))

### Model Selection Recommendations

Generate actionable recommendations based on evaluation results and cost analysis.

In [ ]:
# Calculate overall performance score (simple average of means)
def overall_score(stats: Dict[str, Dict[str, float]]) -> float:
    """Calculate simple average of all mean scores."""
    means = [stat['mean'] for stat in stats.values()]
    return statistics.mean(means) if means else 0.0

overall_nano = overall_score(stats_nano)
overall_gpt5 = overall_score(stats_gpt5)
overall_mini = overall_score(stats_mini)

# Create recommendations
recommendations_md = "## Model Selection Recommendations\n\n"
recommendations_md += "### Overall Performance Scores\n\n"
recommendations_md += f"- **GPT-5 nano**: {overall_nano:.3f}\n"
recommendations_md += f"- **GPT-5**: {overall_gpt5:.3f}\n"
recommendations_md += f"- **GPT-5 mini**: {overall_mini:.3f}\n"
recommendations_md += "\n### Use Case Recommendations\n\n"

# Determine winner
best_model = max([("GPT-5 nano", overall_nano), ("GPT-5", overall_gpt5), ("GPT-5 mini", overall_mini)], key=lambda x: x[1])

recommendations_md += f"**Best Overall Performance**: {best_model[0]}\n\n"
display(Markdown(recommendations_md))

### Summary

You've successfully learned how to perform systematic A/B testing evaluation for agent models. You now understand:

- How to load agent configurations and recreate multiple variants
- Creating diverse evaluation datasets from real-world scenarios
- Running systematic evaluations with multiple built-in evaluators
- Extracting and aggregating evaluation scores across models
- Calculating statistical measures (mean, standard deviation)
- Comparing model performance side-by-side
- Analyzing cost-performance trade-offs
- Generating actionable model selection recommendations

This evaluation methodology enables data-driven decision making for model selection in production systems. By combining performance metrics with cost analysis, you can optimize for both quality and efficiency based on your specific requirements.